# SmartCrop AI: CropHarvest TWO-STEP pipeline (fast)

**Idea:** Step 1 decides *crop vs non-crop* (trained on the large binary dataset, about 87% accurate). Step 2 decides *which crop group* (9 groups). The two are combined into one 10-class prediction.

**Leakage-safe:** Step 1 is trained on the binary pool **minus every 10-class validation sample**, so the 10-class validation and test sets are unseen by both steps.

**Fast:** Step 1 = CNN + BiGRU + LightGBM; Step 2 = 2 CNN + 2 BiGRU + LightGBM; 20 epochs each, printed epoch by epoch. Expected about 6-10 minutes (loading files is the slow part, and it is cached after the first run).

**Honest expectation:** about 74-77% on 10-class validation. 85% is not expected. The notebook prints the real number and compares against the plain 10-class model, so you can see whether the two-step design helps.

In [39]:
# Section 1: Environment Setup & Hardware Inspection
import os
import sys
import json
import time
import math
import random
import shutil
import tempfile
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional
from collections import Counter

# Verify required dependencies
try:
    import numpy as np
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    import h5py
    import sklearn
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
    import matplotlib
    import matplotlib.pyplot as plt
except ImportError as err:
    print(f'[Dependency Warning] Missing package: {err}. Installing...')
    os.system('pip install -q numpy torch h5py scikit-learn matplotlib')
    import numpy as np
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    import h5py
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
    import matplotlib.pyplot as plt

# Fix seeds for exact determinism
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Hardware detection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 80)
print('  SMARTCROP AI: CROPHARVEST COMPLETE SATELLITE TRAINING PIPELINE')
print('=' * 80)
print(f'PyTorch Version  : {torch.__version__}')
print(f'Compute Device   : {DEVICE}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Hardware     : {gpu_name} ({vram_gb:.2f} GB VRAM)')
else:
    print('GPU Hardware     : None detected (Running on CPU)')

# Define Unified Output Hierarchy
if os.path.exists('/kaggle/working'):
    OUTPUT_BASE = Path('/kaggle/working/outputs/cropharvest/complete_training')
else:
    OUTPUT_BASE = Path('outputs/cropharvest/complete_training')

BINARY_OUTPUT_DIR = OUTPUT_BASE / 'binary'
TENCLASS_OUTPUT_DIR = OUTPUT_BASE / '10class'
BINARY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TENCLASS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Base Output Dir  : {OUTPUT_BASE.resolve()}')
print('=' * 80)

  SMARTCROP AI: CROPHARVEST COMPLETE SATELLITE TRAINING PIPELINE
PyTorch Version  : 2.11.0+cu130
Compute Device   : cuda
GPU Hardware     : Tesla T4 (14.56 GB VRAM)
Base Output Dir  : /kaggle/working/outputs/cropharvest/complete_training


In [40]:
# Section 2: Dynamic Dataset Discovery
# Automatically locates CropHarvest files across Kaggle input directories and local paths.

from pathlib import Path
import os

def find_cropharvest_root() -> Path:
    search_roots = [
        Path('/kaggle/input'),
        Path('/kaggle/working'),
        Path('datasets/cropharvest'),
        Path('../datasets/cropharvest'),
        Path('../../datasets/cropharvest'),
        Path('.')
    ]

    # Search direct and nested directories recursively
    for root in search_roots:
        if not root.exists():
            continue

        # Search for labels.geojson and verify required files
        for geojson in root.rglob('labels.geojson'):
            candidate = geojson.parent

            if (
                (candidate / 'normalizing_dict.h5').is_file()
                and (candidate / 'arrays').is_dir()
            ):
                return candidate

    raise FileNotFoundError(
        'Could not locate CropHarvest dataset root containing '
        'labels.geojson, normalizing_dict.h5 and arrays/. '
        'Please check that the dataset is attached to Kaggle.'
    )


# Locate dataset
DATASET_ROOT = find_cropharvest_root()

GEOJSON_FILE = DATASET_ROOT / 'labels.geojson'
NORM_DICT_FILE = DATASET_ROOT / 'normalizing_dict.h5'
ARRAYS_DIR = DATASET_ROOT / 'arrays'

# Verify required paths
assert GEOJSON_FILE.is_file(), 'labels.geojson missing'
assert NORM_DICT_FILE.is_file(), 'normalizing_dict.h5 missing'
assert ARRAYS_DIR.is_dir(), 'arrays directory missing'

print('Dataset Root Located : ', DATASET_ROOT.resolve())
print('GeoJSON Labels Path  : ', GEOJSON_FILE.resolve())
print('Normalizing Dict Path: ', NORM_DICT_FILE.resolve())
print('Arrays Directory Path: ', ARRAYS_DIR.resolve())

print('\nDataset discovery successful!')

Dataset Root Located :  /kaggle/input/datasets/sree0987/smartcrop-cropharvest/cropharvest
GeoJSON Labels Path  :  /kaggle/input/datasets/sree0987/smartcrop-cropharvest/cropharvest/labels.geojson
Normalizing Dict Path:  /kaggle/input/datasets/sree0987/smartcrop-cropharvest/cropharvest/normalizing_dict.h5
Arrays Directory Path:  /kaggle/input/datasets/sree0987/smartcrop-cropharvest/cropharvest/arrays

Dataset discovery successful!


In [41]:
# Section 2: Dataset Integrity & Feature Shape Audit
# Verifies 18 channels, 12 monthly timesteps, and 0 NaNs/Infs.

with h5py.File(NORM_DICT_FILE, 'r') as h:
    norm_mean = h['mean'][:].astype(np.float32)
    norm_std = h['std'][:].astype(np.float32)

print(f'Normalizing Mean Shape : {norm_mean.shape} (Expected: (18,))')
print(f'Normalizing Std Shape  : {norm_std.shape} (Expected: (18,))')
assert norm_mean.shape == (18,), f'Invalid mean shape: {norm_mean.shape}'
assert norm_std.shape == (18,), f'Invalid std shape: {norm_std.shape}'

with open(GEOJSON_FILE, 'r', encoding='utf-8') as f:
    geo_data = json.load(f)

all_features = geo_data['features']
print(f'Total Features in GeoJSON: {len(all_features):,}')

# Inspect available HDF5 files using fast os.listdir
available_array_files = set(f for f in os.listdir(ARRAYS_DIR) if f.endswith('.h5'))
print(f'Available HDF5 Arrays    : {len(available_array_files):,}')

# Inspect sample array integrity
sample_filename = next(iter(available_array_files))
sample_file = ARRAYS_DIR / sample_filename
with h5py.File(sample_file, 'r') as h:
    sample_arr = h['array'][:]
    print(f'Sample File Name         : {sample_file.name}')
    print(f'Sample Array Shape       : {sample_arr.shape} (Expected: (12, 18))')
    print(f'Sample Array Dtype       : {sample_arr.dtype}')
    print(f'Sample Array NaNs        : {np.isnan(sample_arr).sum()}')
    print(f'Sample Array Infs        : {np.isinf(sample_arr).sum()}')

assert sample_arr.shape == (12, 18), f'Unexpected array shape: {sample_arr.shape}'
print('[Audit Passed] Dataset format, 18 channels, and 12-month temporal dimensions verified.')

Normalizing Mean Shape : (18,) (Expected: (18,))
Normalizing Std Shape  : (18,) (Expected: (18,))
Total Features in GeoJSON: 113,893
Available HDF5 Arrays    : 87,464
Sample File Name         : 3831_ile-de-france.h5
Sample Array Shape       : (12, 18) (Expected: (12, 18))
Sample Array Dtype       : float64
Sample Array NaNs        : 0
Sample Array Infs        : 0
[Audit Passed] Dataset format, 18 channels, and 12-month temporal dimensions verified.


In [42]:
# Section 3: Official Split Verification & Geographic Assessment
# Preserves official is_test=true samples as held-out test data.
# Partitions is_test=false into 80% Train and 20% Validation (SEED=42).
# Audits zero-overlap and documents spatial autocorrelation limitations.

# Define Official 10-Class Taxonomy
CLASSES_10 = [
    'cereals', 'other', 'fruits_nuts', 'vegetables_melons',
    'oilseeds', 'leguminous', 'root_tuber', 'beverage_spice', 'sugar',
    'non_crop'
]
CLASS_TYPES_10 = {
    'cereals': 'Crop Category',
    'other': 'Crop Category',
    'fruits_nuts': 'Crop Category',
    'vegetables_melons': 'Crop Category',
    'oilseeds': 'Crop Category',
    'leguminous': 'Crop Category',
    'root_tuber': 'Crop Category',
    'beverage_spice': 'Crop Category',
    'sugar': 'Crop Category',
    'non_crop': 'Land-Cover Class (Non-Agricultural)'
}
CLS_TO_IDX_10 = {c: i for i, c in enumerate(CLASSES_10)}
IDX_TO_CLS_10 = {i: c for i, c in enumerate(CLASSES_10)}

# 1. Parse all matched features
arrays_dir_resolved = ARRAYS_DIR.resolve()
records_binary = []
records_10class = []

for feat in all_features:
    props = feat['properties']
    fn = f"{props['index']}_{props['dataset']}.h5"
    if fn in available_array_files:
        rec = {
            'filename': fn,
            'filepath': str(arrays_dir_resolved / fn),
            'index': props['index'],
            'dataset': props['dataset'],
            'lat': float(props['lat']),
            'lon': float(props['lon']),
            'is_crop': int(props['is_crop']),
            'classification_label': props.get('classification_label'),
            'is_test_official': bool(props.get('is_test', False))
        }
        records_binary.append(rec)
        if rec['classification_label'] in CLS_TO_IDX_10:
            rec_10 = dict(rec)
            rec_10['class_idx'] = CLS_TO_IDX_10[rec['classification_label']]
            rec_10['class_type'] = CLASS_TYPES_10[rec['classification_label']]
            records_10class.append(rec_10)

print(f'Total Matched Binary Records   : {len(records_binary):,}')
print(f'Total Matched 10-Class Records : {len(records_10class):,}')

# 2. Partition Binary Splits (is_test=False -> 80% Train, 20% Val)
bin_pool = [r for r in records_binary if not r['is_test_official']]
bin_test = [r for r in records_binary if r['is_test_official']]
random.seed(RANDOM_SEED)
random.shuffle(bin_pool)
bin_val_size = int(len(bin_pool) * 0.20)
bin_val = bin_pool[:bin_val_size]
bin_train = bin_pool[bin_val_size:]

# 3. Partition 10-Class Splits (Per-class stratified 80:20)
c10_pool = [r for r in records_10class if not r['is_test_official']]
c10_test = [r for r in records_10class if r['is_test_official']]
c10_train = []
c10_val = []
random.seed(RANDOM_SEED)
for c in CLASSES_10:
    c_recs = [r for r in c10_pool if r['classification_label'] == c]
    random.shuffle(c_recs)
    n_v = int(len(c_recs) * 0.20)
    c10_val.extend(c_recs[:n_v])
    c10_train.extend(c_recs[n_v:])

# 4. Zero-Overlap Audit
bin_train_ids = set(r['filename'] for r in bin_train)
bin_val_ids = set(r['filename'] for r in bin_val)
bin_test_ids = set(r['filename'] for r in bin_test)
assert len(bin_train_ids.intersection(bin_val_ids)) == 0, 'Binary Train/Val overlap detected!'
assert len(bin_train_ids.intersection(bin_test_ids)) == 0, 'Binary Train/Test overlap detected!'
assert len(bin_val_ids.intersection(bin_test_ids)) == 0, 'Binary Val/Test overlap detected!'

c10_train_ids = set(r['filename'] for r in c10_train)
c10_val_ids = set(r['filename'] for r in c10_val)
c10_test_ids = set(r['filename'] for r in c10_test)
assert len(c10_train_ids.intersection(c10_val_ids)) == 0, '10-Class Train/Val overlap detected!'
assert len(c10_train_ids.intersection(c10_test_ids)) == 0, '10-Class Train/Test overlap detected!'
assert len(c10_val_ids.intersection(c10_test_ids)) == 0, '10-Class Val/Test overlap detected!'

print('\n' + '=' * 80)
print('  AUTHORITATIVE DATASET SPLIT SUMMARY')
print('=' * 80)
print(f'Binary Task    : Train={len(bin_train):,} | Val={len(bin_val):,} | Official Test={len(bin_test):,} | Sum={len(records_binary):,}')
print(f'10-Class Task  : Train={len(c10_train):,} | Val={len(c10_val):,} | Official Test={len(c10_test):,}   | Sum={len(records_10class):,}')
print('=' * 80)
print('[Spatial Autocorrelation Disclosure]')
print('Internal validation splits use random per-class stratification across 16 shared regional datasets.')
print('This validates temporal phenology modeling, NOT zero-shot spatial transferability.')
print('True geographic domain generalization is measured on the official held-out test split.')
print('=' * 80)

Total Matched Binary Records   : 87,464
Total Matched 10-Class Records : 42,603

  AUTHORITATIVE DATASET SPLIT SUMMARY
Binary Task    : Train=66,124 | Val=16,530 | Official Test=4,810 | Sum=87,464
10-Class Task  : Train=34,000 | Val=8,496 | Official Test=107   | Sum=42,603
[Spatial Autocorrelation Disclosure]
Internal validation splits use random per-class stratification across 16 shared regional datasets.
This validates temporal phenology modeling, NOT zero-shot spatial transferability.
True geographic domain generalization is measured on the official held-out test split.


In [43]:
# Cell 5: Load all arrays (cached after the first run)
import torch.nn.functional as F
from concurrent.futures import ProcessPoolExecutor
try:
    import lightgbm as lgb
except ImportError:
    os.system('pip install -q lightgbm'); import lightgbm as lgb
import pandas as pd

def _read_chunk(paths):
    out = np.empty((len(paths), 12, 18), np.float32)
    for i, p in enumerate(paths):
        with h5py.File(p, 'r') as h: out[i] = h['array'][:]
    return out

t0 = time.time(); CACHE = OUTPUT_BASE / 'all_arrays_normalized.npy'
paths = [r['filepath'] for r in records_binary]
if CACHE.exists() and np.load(CACHE, mmap_mode='r').shape[0] == len(paths):
    ALL = np.load(CACHE); print('Loaded from cache.')
else:
    chunks = [paths[i:i + 400] for i in range(0, len(paths), 400)]
    with ProcessPoolExecutor(max_workers=8) as ex: parts = list(ex.map(_read_chunk, chunks))
    ALL = np.concatenate(parts)
    ALL = np.nan_to_num((ALL - norm_mean) / np.where(norm_std == 0, 1.0, norm_std), nan=0.0, posinf=0.0, neginf=0.0)
    ALL = np.clip(ALL, -10, 10).astype(np.float32); np.save(CACHE, ALL)
path_idx = {p: i for i, p in enumerate(paths)}
print(f'Arrays {ALL.shape} ready in {time.time()-t0:.0f}s')

def XY(recs, key):
    return ALL[[path_idx[r['filepath']] for r in recs]], np.array([r[key] for r in recs], dtype=np.int64)

Arrays (87464, 12, 18) ready in 36s


In [44]:
# Cell 6: Four model families + training helpers
class TSTransformer(nn.Module):
    def __init__(self, ncls, nin=18, d=96, heads=4, layers=3, drop=0.2):
        super().__init__()
        self.inp = nn.Linear(nin, d)
        self.pos = nn.Parameter(torch.zeros(1, 12, d)); nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(d, heads, d * 2, drop, batch_first=True, norm_first=True, activation='gelu')
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(0.3), nn.Linear(d, ncls))
    def forward(self, x):
        h = self.norm(self.enc(self.inp(x) + self.pos))
        return self.head(torch.cat([h.mean(1), h.amax(1)], 1))

class CNN1D(nn.Module):
    def __init__(self, ncls, nin=18, d=96):
        super().__init__()
        def blk(i, o): return nn.Sequential(nn.Conv1d(i, o, 3, padding=1), nn.BatchNorm1d(o), nn.GELU())
        self.b1, self.b2, self.b3 = blk(nin, d), blk(d, d), blk(d, d)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(0.3), nn.Linear(d, ncls))
    def forward(self, x):
        h = self.b1(x.transpose(1, 2)); h = h + self.b2(h); h = h + self.b3(h)
        return self.head(torch.cat([h.mean(2), h.amax(2)], 1))

class BiGRU(nn.Module):
    def __init__(self, ncls, nin=18, d=96):
        super().__init__()
        self.rnn = nn.GRU(nin, d, 2, batch_first=True, bidirectional=True, dropout=0.2)
        self.norm = nn.LayerNorm(2 * d)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(4 * d, d), nn.GELU(), nn.Dropout(0.3), nn.Linear(d, ncls))
    def forward(self, x):
        h = self.norm(self.rnn(x)[0])
        return self.head(torch.cat([h.mean(1), h.amax(1)], 1))

USE_CUDA = DEVICE.type == 'cuda'
AC = lambda: torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_CUDA)

@torch.no_grad()
def nn_probs(net, X, bs=2048):
    net.eval(); out = []
    for i in range(0, len(X), bs):
        with AC(): out.append(F.softmax(net(X[i:i + bs]).float(), 1).cpu())
    return torch.cat(out).numpy()

def train_nn(make_net, Xtr, ytr, ncls, seed, epochs=30, bs=256, lr=2e-3, wd=0.05, ema_decay=0.995, name='', val=None, every=1):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    net = make_net(ncls).to(DEVICE)
    ema = {k: v.detach().clone() for k, v in net.state_dict().items()}
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)
    spe = math.ceil(len(Xtr) / bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * spe, pct_start=0.15)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_CUDA)
    for ep in range(epochs):
        net.train(); perm = torch.randperm(len(Xtr), device=DEVICE); run, cnt, te = 0.0, 0, time.time()
        for i in range(spe):
            idx = perm[i * bs:(i + 1) * bs]
            if len(idx) < 8: continue
            x, y = Xtr[idx], ytr[idx]
            x = (x + 0.03 * torch.randn_like(x)) * (torch.rand_like(x) > 0.05)
            lam, yb = 1.0, y
            if random.random() < 0.5:                                   # sequence MixUp
                lam = float(np.random.beta(0.4, 0.4)); j = torch.randperm(len(x), device=DEVICE)
                x, yb = lam * x + (1 - lam) * x[j], y[j]
            opt.zero_grad(set_to_none=True)
            with AC():
                out = net(x).float()
                loss = lam * F.cross_entropy(out, y, label_smoothing=0.1) + (1 - lam) * F.cross_entropy(out, yb, label_smoothing=0.1)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item() * len(idx); cnt += len(idx)
            with torch.no_grad():
                for k, v in net.state_dict().items():
                    if v.dtype.is_floating_point: ema[k].mul_(ema_decay).add_(v.detach(), alpha=1 - ema_decay)
                    else: ema[k].copy_(v)
        if val is not None and ((ep + 1) % every == 0 or ep == epochs - 1):     # progress display only (not used for selection)
            va_acc = float((nn_probs(net, val[0]).argmax(1) == val[1]).mean())
            print(f'    [{name}] Epoch {ep+1:02d}/{epochs} | train loss {run/max(1,cnt):.4f} | val acc {va_acc*100:.2f}% | {time.time()-te:.1f}s')
    net.load_state_dict(ema)                                           # fixed epochs, EMA weights: no val-based epoch picking
    return net

def lgb_feats(X):
    return np.concatenate([X.reshape(len(X), -1), X.mean(1), X.std(1), X.min(1), X.max(1), np.diff(X, axis=1).reshape(len(X), -1)], 1)

def metrics(y, p):
    pred = p.argmax(1); m = {'accuracy': float((pred == y).mean())}
    for avg in ('macro', 'weighted'):
        P, R, Fs, _ = precision_recall_fscore_support(y, pred, average=avg, zero_division=0)
        m[f'{avg}_precision'], m[f'{avg}_recall'], m[f'{avg}_f1'] = float(P), float(R), float(Fs)
    return m
print('Helpers ready.')

Helpers ready.


In [45]:
# Cell 7: STEP 1 - crop vs non-crop (trained WITHOUT any 10-class validation sample)
c10_val_ids = {r['filename'] for r in c10_val}
pool1 = [r for r in bin_pool if r['filename'] not in c10_val_ids]
random.Random(42).shuffle(pool1)
n_h = int(0.1 * len(pool1)); s1_hold, s1_train = pool1[:n_h], pool1[n_h:]
assert not ({r['filename'] for r in s1_train} | {r['filename'] for r in s1_hold}) & c10_val_ids, 'LEAK: 10-class val in step 1 data'
print(f'Step 1: train={len(s1_train):,} holdout={len(s1_hold):,} (removed {len(bin_pool)-len(pool1):,} samples that belong to the 10-class validation set)')

X1tr, y1tr = XY(s1_train, 'is_crop'); X1h, y1h = XY(s1_hold, 'is_crop'); X1te, y1te = XY(bin_test, 'is_crop')
Xc_tr, yc_tr = XY(c10_train, 'class_idx'); Xc_va, yc_va = XY(c10_val, 'class_idx'); Xc_te, yc_te = XY(c10_test, 'class_idx')
T = lambda a: torch.from_numpy(a).to(DEVICE)
g1tr, y1tr_t = T(X1tr), T(y1tr)
EVAL1 = {'hold': X1h, 'btest': X1te, 'c_train': Xc_tr, 'c_val': Xc_va, 'c_test': Xc_te}
EVAL1_G = {k: T(v) for k, v in EVAL1.items()}
EP = 20; t0 = time.time(); fam1 = {}
for name, cls in (('CNN1D', CNN1D), ('BiGRU', BiGRU)):
    print(f'--- STEP 1: training {name} ({EP} epochs) ---')
    net = train_nn(cls, g1tr, y1tr_t, 2, seed=7, epochs=EP, name=f'S1 {name}', val=(EVAL1_G['hold'], y1h), every=2)
    fam1[name] = {k: nn_probs(net, g) for k, g in EVAL1_G.items()}
    del net; torch.cuda.empty_cache()
print('--- STEP 1: LightGBM ---')
clf1 = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.08, num_leaves=31, min_child_samples=30, reg_lambda=5.0, subsample=0.8,
                          subsample_freq=1, colsample_bytree=0.5, n_jobs=4, random_state=42, verbose=-1)
clf1.fit(lgb_feats(X1tr), y1tr, eval_set=[(lgb_feats(X1h), y1h)], callbacks=[lgb.early_stopping(40, verbose=False)])
fam1['LightGBM'] = {k: clf1.predict_proba(lgb_feats(X)) for k, X in EVAL1.items()}
S1 = {k: sum(f[k] for f in fam1.values()) / len(fam1) for k in EVAL1}          # equal weight per family
for n, f in fam1.items(): print(f'  step-1 {n:<9s} holdout acc {metrics(y1h, f["hold"])["accuracy"]*100:.2f}%')
m_h, m_t = metrics(y1h, S1['hold']), metrics(y1te, S1['btest'])
print(f'\nSTEP 1 ensemble | holdout acc {m_h["accuracy"]*100:.2f}% (F1 macro {m_h["macro_f1"]*100:.2f}%) | official test acc {m_t["accuracy"]*100:.2f}% (F1 macro {m_t["macro_f1"]*100:.2f}%) | {time.time()-t0:.0f}s')

Step 1: train=66,743 holdout=7,415 (removed 8,496 samples that belong to the 10-class validation set)
--- STEP 1: training CNN1D (20 epochs) ---
    [S1 CNN1D] Epoch 02/20 | train loss 0.5167 | val acc 81.94% | 1.7s
    [S1 CNN1D] Epoch 04/20 | train loss 0.4939 | val acc 82.97% | 1.7s
    [S1 CNN1D] Epoch 06/20 | train loss 0.4821 | val acc 83.91% | 1.7s
    [S1 CNN1D] Epoch 08/20 | train loss 0.4709 | val acc 83.95% | 1.7s
    [S1 CNN1D] Epoch 10/20 | train loss 0.4680 | val acc 84.41% | 1.7s
    [S1 CNN1D] Epoch 12/20 | train loss 0.4570 | val acc 85.23% | 1.7s
    [S1 CNN1D] Epoch 14/20 | train loss 0.4496 | val acc 85.37% | 1.6s
    [S1 CNN1D] Epoch 16/20 | train loss 0.4464 | val acc 85.53% | 1.6s
    [S1 CNN1D] Epoch 18/20 | train loss 0.4412 | val acc 85.87% | 1.7s
    [S1 CNN1D] Epoch 20/20 | train loss 0.4409 | val acc 86.08% | 1.7s
--- STEP 1: training BiGRU (20 epochs) ---
    [S1 BiGRU] Epoch 02/20 | train loss 0.5198 | val acc 80.20% | 2.1s
    [S1 BiGRU] Epoch 04/20 | tr

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  step-1 CNN1D     holdout acc 86.15%
  step-1 BiGRU     holdout acc 85.83%
  step-1 LightGBM  holdout acc 86.11%

STEP 1 ensemble | holdout acc 86.68% (F1 macro 84.63%) | official test acc 82.43% (F1 macro 81.63%) | 114s


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [46]:
# Cell 8: STEP 2 - which crop group (plain 10-class model, trained on 10-class train only)
gc_tr, yc_tr_t = T(Xc_tr), T(yc_tr)
EVAL2 = {'train': Xc_tr, 'val': Xc_va, 'test': Xc_te}; EVAL2_G = {k: T(v) for k, v in EVAL2.items()}
t0 = time.time(); fam2 = {}
for name, cls, n in (('CNN1D', CNN1D, 2), ('BiGRU', BiGRU, 2)):
    acc = {k: 0 for k in EVAL2}
    for s in range(n):
        print(f'--- STEP 2: training {name} {s+1}/{n} ({EP} epochs) ---')
        net = train_nn(cls, gc_tr, yc_tr_t, 10, seed=100 + s, epochs=EP, name=f'S2 {name} #{s+1}', val=(EVAL2_G['val'], yc_va), every=(2 if s == 0 else 10))
        for k, g in EVAL2_G.items(): acc[k] = acc[k] + nn_probs(net, g) / n
        del net; torch.cuda.empty_cache()
    fam2[name] = acc
print('--- STEP 2: LightGBM ---')
clf2 = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.08, num_leaves=31, min_child_samples=30, reg_lambda=5.0, subsample=0.8,
                          subsample_freq=1, colsample_bytree=0.5, n_jobs=4, random_state=42, verbose=-1)
clf2.fit(lgb_feats(Xc_tr), yc_tr, eval_set=[(lgb_feats(Xc_va), yc_va)], callbacks=[lgb.early_stopping(40, verbose=False)])
fam2['LightGBM'] = {k: clf2.predict_proba(lgb_feats(X)) for k, X in EVAL2.items()}
S2 = {k: sum(f[k] for f in fam2.values()) / len(fam2) for k in EVAL2}
for n, f in fam2.items(): print(f'  step-2 {n:<9s} val acc {metrics(yc_va, f["val"])["accuracy"]*100:.2f}%')
print(f'\nPlain 10-class ensemble (no step 1): val acc {metrics(yc_va, S2["val"])["accuracy"]*100:.2f}%  | {time.time()-t0:.0f}s')

--- STEP 2: training CNN1D 1/2 (20 epochs) ---
    [S2 CNN1D #1] Epoch 02/20 | train loss 1.6564 | val acc 58.76% | 0.9s
    [S2 CNN1D #1] Epoch 04/20 | train loss 1.4931 | val acc 64.34% | 0.9s
    [S2 CNN1D #1] Epoch 06/20 | train loss 1.4429 | val acc 66.78% | 0.9s
    [S2 CNN1D #1] Epoch 08/20 | train loss 1.3906 | val acc 68.34% | 0.8s
    [S2 CNN1D #1] Epoch 10/20 | train loss 1.3565 | val acc 68.77% | 0.8s
    [S2 CNN1D #1] Epoch 12/20 | train loss 1.3501 | val acc 69.77% | 0.9s
    [S2 CNN1D #1] Epoch 14/20 | train loss 1.3094 | val acc 70.41% | 0.8s
    [S2 CNN1D #1] Epoch 16/20 | train loss 1.2625 | val acc 71.22% | 0.9s
    [S2 CNN1D #1] Epoch 18/20 | train loss 1.2998 | val acc 71.28% | 0.9s
    [S2 CNN1D #1] Epoch 20/20 | train loss 1.2858 | val acc 71.27% | 0.8s
--- STEP 2: training CNN1D 2/2 (20 epochs) ---
    [S2 CNN1D #2] Epoch 10/20 | train loss 1.3815 | val acc 68.80% | 0.9s
    [S2 CNN1D #2] Epoch 20/20 | train loss 1.3092 | val acc 71.17% | 0.9s
--- STEP 2: traini

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  step-2 CNN1D     val acc 71.56%
  step-2 BiGRU     val acc 71.99%
  step-2 LightGBM  val acc 73.68%

Plain 10-class ensemble (no step 1): val acc 73.49%  | 163s


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [47]:
# Cell 9: Combine the two steps -> final 10-class result
NC = CLASSES_10.index('non_crop'); CROP = [i for i in range(10) if i != NC]
def combine(p_step1, p_step2):
    pc = p_step1[:, 1:2]                                                    # P(crop) from step 1
    crop_part = p_step2[:, CROP]; crop_part = crop_part / (crop_part.sum(1, keepdims=True) + 1e-9)
    out = np.zeros_like(p_step2); out[:, CROP] = pc * crop_part; out[:, NC] = 1 - pc[:, 0]
    return out
FINAL = {'train': combine(S1['c_train'], S2['train']), 'val': combine(S1['c_val'], S2['val']), 'test': combine(S1['c_test'], S2['test'])}
ys = {'train': yc_tr, 'val': yc_va, 'test': yc_te}
M = {'Training': metrics(yc_tr, FINAL['train']), 'Validation': metrics(yc_va, FINAL['val']), 'Official Test': metrics(yc_te, FINAL['test'])}
rows = [('Accuracy', 'accuracy'), ('Precision (macro)', 'macro_precision'), ('Recall (macro)', 'macro_recall'), ('F1 (macro)', 'macro_f1'),
        ('Precision (weighted)', 'weighted_precision'), ('Recall (weighted)', 'weighted_recall'), ('F1 (weighted)', 'weighted_f1')]
table = pd.DataFrame({k: {l: round(v[key] * 100, 2) for l, key in rows} for k, v in M.items()})
plain = metrics(yc_va, S2['val']); two = M['Validation']
print('=' * 74); print('TWO-STEP 10-CLASS RESULT'); print('=' * 74); print(table.to_string())
print(f'\nValidation accuracy: plain 10-class {plain["accuracy"]*100:.2f}%  ->  two-step {two["accuracy"]*100:.2f}%   (macro-F1 {plain["macro_f1"]*100:.2f}% -> {two["macro_f1"]*100:.2f}%)')
print(f'85% target: {"REACHED" if two["accuracy"] >= 0.85 else "not reached"}')

pred = FINAL['val'].argmax(1)
p, r, f, s = precision_recall_fscore_support(yc_va, pred, labels=list(range(10)), zero_division=0)
rep = pd.DataFrame({'class': CLASSES_10, 'precision': p.round(3), 'recall': r.round(3), 'f1': f.round(3), 'val_samples': s})
print('\nPer-class validation report:'); print(rep.to_string(index=False))
OUT_DIR = TENCLASS_OUTPUT_DIR
table.to_csv(OUT_DIR / 'two_step_metrics_table.csv'); rep.to_csv(OUT_DIR / 'two_step_per_class_val.csv', index=False)
json.dump({'step1_holdout': m_h, 'step1_official_test': m_t, 'two_step': M, 'plain_val': plain}, open(OUT_DIR / 'two_step_metrics.json', 'w'), indent=2)
print('\nOfficial 10-class test has only', len(yc_te), 'samples - do not rely on its macro scores.')

TWO-STEP 10-CLASS RESULT
                      Training  Validation  Official Test
Accuracy                 87.21       73.19          56.07
Precision (macro)        88.98       71.11          34.69
Recall (macro)           84.30       62.82          24.37
F1 (macro)               86.27       65.14          23.82
Precision (weighted)     87.43       72.90          51.91
Recall (weighted)        87.21       73.19          56.07
F1 (weighted)            87.10       72.33          47.39

Validation accuracy: plain 10-class 73.49%  ->  two-step 73.19%   (macro-F1 65.47% -> 65.14%)
85% target: not reached

Per-class validation report:
            class  precision  recall    f1  val_samples
          cereals      0.769   0.895 0.827         2647
            other      0.798   0.712 0.753         1679
      fruits_nuts      0.684   0.731 0.707          821
vegetables_melons      0.667   0.708 0.687          751
         oilseeds      0.626   0.417 0.501          573
       leguminous      0.6